# Veri, Etiketleme ve Referans Etiketler — eşlik eden notebook

*Uygulamalı Doğal Dil İşleme — Token'lardan Ajanlara* kitabının 5. bölümü.
Önce bölümü okuyun: **[Veri, Etiketleme ve Referans Etiketler](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html)**.

Önceki bölüm temsilleri `sentiment` adlı bir sütuna karşı karşılaştırdı. Bu bölüm o
sütunun nereden geldiğini ve bir kısmı yanlışsa skorların ne anlama geldiğini soruyor.
Önceki bölümün bölmesini ve en yüksek skorlu modelini sabit tutup etiketleri değiştiriyor.
Bu notebook o deneyleri bölümün sırasıyla çalıştırıyor:

1. **Dondurulmuş model**: önceki bölümün bölmesi ve kelime unigram + bigram TF-IDF satırı.
2. **Yüz satır okuyun**: bölümün elle okuduğu rastgele örneklem ve üç kuralının aritmetiği.
3. **Şema**: iki hüküm barındırıyor olabilecek yorumlar için dört kelimelik bir ön eleme
   kuralı.
4. **Uyum**: tek bir 2×2 tablodan ham uyum, Cohen kappa'sı ve üç katsayı daha; ve sınıf
   açısından dengelenmiş bir altörneklemin bunlara yaptığı.
5. **Etiketleri ölçekte denetlemek**: out-of-fold işaretler, işaretlenen satırları silmenin
   ya da çevirmenin etkisi ve kontrol olarak aynı işlemin rastgele satırlara uygulanması.
6. **Eğitimde etiket gürültüsü**, yöne ve sayıya göre.
7. **Test kümesinde etiket gürültüsü** ve iki model arasındaki küçük bir farkın anlamı.
8. **Sızıntı**: birebir ve yakın yinelemeler, ve kontrollü bulaşma.
9. **Bütçe**: az sayıda temiz etikete karşı çok sayıda gürültülü etiket, ve belirsizlik
   örneklemesi.

**Sayılar nasıl okunur.** Aşağıda basılan her şey *bu koşunun* ürettiği şey. Bölüm aynı
niceliği basıyorsa hücre kitabın değerini yanına yazıyor. Üç tür karşılaştırma var ve her
hücre hangisinin geçerli olduğunu söylüyor:

- *deterministik* nicelikler (sayımlar, dondurulmuş model, kodunu bölümün bastığı ya da
  birebir tarif ettiği sabit seed'li deneyler) kitabın bastığı hassasiyette
  karşılaştırılıyor;
- *bölümün seed'lerini basmadığı rastgele çekimli deneyler* (gürültü tabloları, bütçe
  ızgarası, belirsizlik örneklemesi) bu notebook'un kendi seed'leriyle koşuluyor ve hücre
  kitabın değerine olan uzaklığı seed'ler arası yayılımın yanında bildiriyor;
- *insan yargıları* (100 satırın hangisinin hata olduğu, 40 itirazın hangisinin şema
  sorunu olduğu) kodla yeniden üretilemez. Notebook size aynı satırları gösteriyor ve
  bölümün sayımlarını bölümün sayımları olarak veriyor.

**Bu notebook'un yeniden üretemediği bir şey var.** Bölümün ikinci "etiketleyicisi", 36
olumlu ve 37 olumsuz ifadeden oluşan bir sözlük kuralı ve bölüm bu listeleri basmıyor. Bu
yüzden kuralın 2×2 tablosu veri olarak alınıyor ve ondan türeyen her katsayı yeniden
hesaplanıyor. O kısmın baştan sona çalıştırılabilmesi için notebook kendi küçük kuralını
da tanımlıyor (22 ifade; 1. bölümün notebook'undaki kural) ve analizi onun üzerinde
tekrarlıyor. O kuralın sayıları bölümün sayıları değil.

**Dil hakkında bir not.** Anlatım ve kod yorumları Türkçe; kodun kendisi ve *basılan*
etiketler İngilizce (`same as the book`, `DIFFERS`, `not comparable` gibi). Sebebi, Türkçe
bölümün kendi `text` bloklarının da İngilizce olması: burada bastığınız tabloyu kitaptaki
tabloyla yan yana okuyabilirsiniz.

**Süre ve indirmeler.** Bir dizüstü CPU'sunda (Apple M serisi) yaklaşık 2 dakika. Tek
indirme veri kümesi (24 MB); bu notebook'un yanındaki `data/cache/` altında önbelleğe
alınıyor. Model seyrek öznitelikler üzerinde doğrusal bir sınıflandırıcı; bu yüzden onu
yüzden fazla kez yeniden eğiten deneyler bile saniyeler sürüyor. Ağır hücre yok ve diğer
notebook'lardaki `ANLP_SKIP_HEAVY` anahtarı burada hiçbir şeyi değiştirmiyor.

## 1. Kurulum

İlk hücre bu koşunun kullandığı yorumlayıcıyı ve paket sürümlerini kaydediyor. Sizin
sayılarınız bu notebook'ta kayıtlı olanlardan farklıysa önce bu hücreyi karşılaştırın.

In [1]:
import hashlib
import platform
import re
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
from scipy.stats import binom
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scipy", "scikit-learn", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scipy         1.18.1
scikit-learn  1.9.0
pyarrow       25.0.1
requests      2.34.2


Baştan sona kullanılan üç yardımcı.

- `vs_book` bu koşunun sayısını kitabınkinin yanına basıyor ve ikisini kitabın bastığı
  hassasiyette karşılaştırıyor.
- `vs_book_near` bu notebook'un kendi rastgele çekimleriyle koşulan deneyler için.
  Kitabın değerinden farkı basıyor; seed'ler arası yayılım biliniyorsa farkı onunla
  birlikte okuyun.
- `scores` doğruluğu (accuracy), macro-F1'i ve olumsuz sınıfın F1'ini yüzde olarak döndürüyor.

In [2]:
def vs_book(label, ours, book, fmt="{}", note=None):
    """Bu koşunun değeri kitabınkinin yanında; kitabın bastığı hassasiyette karşılaştırılır."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    if note:
        verdict = f"not comparable: {note}"
    else:
        verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<40} this run {ours_s:>10}   book {book_s:>10}   {verdict}")


def vs_book_near(label, ours, book, fmt="{:.2f}", note=None):
    """Bu notebook'un kendi rastgele çekimleriyle koşulan deneyler için: kitabın değerine olan uzaklık."""
    verdict = f"not comparable: {note}" if note else f"{ours - book:+.2f} from the book"
    print(f"{label:<40} this run {fmt.format(ours):>10}   book {fmt.format(book):>10}   {verdict}")


def scores(y_true, y_pred):
    """Doğruluk (accuracy), macro-F1 ve olumsuz sınıfın F1'i, yüzde olarak."""
    return (100 * accuracy_score(y_true, y_pred),
            100 * f1_score(y_true, y_pred, average="macro", zero_division=0),
            100 * f1_score(y_true, y_pred, pos_label=0, zero_division=0))


def macro_f1(y_true, y_pred):
    return 100 * f1_score(y_true, y_pred, average="macro", zero_division=0)

## 2. Şimdiye kadar güvendiğiniz sayı

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#şimdiye-kadar-güvendiğiniz-sayı)

Veri, bölme ve model önceki bölümünkiler, değişmeden: `fthbrmnby/turkish_product_reviews`
Türkçe ürün yorumları (CC BY-SA 4.0), tekilleştirmeden sonra 50.000 yorumluk tabakalı bir
örneklem, seed 42 ile 40.000 / 10.000 bölme, ve kelime unigram ile bigram'ları üzerinde
TF-IDF ile sınıf ağırlıklı lojistik regresyon. [Önceki bölümün notebook'u](https://github.com/bbardakk/uygulamali-dogal-dil-isleme-notebooks/blob/main/tr/chapters/04-karsilastirma/04-karsilastirma.ipynb) bunların her
birini adım adım kuruyor; burada hepsi tek hücre.

Bölüm `sentiment` sütunundaki etiketlere **referans etiketler** diyor: değerlendirmede
başvuru noktası; metnin tartışmasız tek yorumu olduğu iddiası değil.

**Neye bakacaksınız:** parmak izleri elinizde önceki bölümle aynı satırların olduğunu
söylüyor; üç skor da aşağıdaki her deneyin başladığı satır.

In [3]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)

# 4. bölümün yükleme kodu ve bölmesi
raw = pd.read_parquet(CACHE).rename(columns={"sentence": "text", "sentiment": "label"})
raw["text"] = raw["text"].astype(str).str.strip()
df = raw.drop_duplicates(subset="text")
sub, _ = train_test_split(df, train_size=50_000, stratify=df["label"], random_state=42)
train, test = train_test_split(sub, test_size=0.2, stratify=sub["label"], random_state=42)
y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()

def fingerprint(frame):
    rows = ",".join(map(str, sorted(frame.index)))
    return hashlib.sha256(rows.encode()).hexdigest()[:12]

# 4. bölümün tokenleştirmesi ve en yüksek skorlu satırı
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})
def tr_lower(s):
    return s.translate(TR_LOWER).lower()

PAT = r"[a-zçğıöşü]+"
LOC = dict(preprocessor=tr_lower, lowercase=False)
TOKEN_RE = re.compile(PAT)
def tokenize(s):
    return TOKEN_RE.findall(tr_lower(s))

def make_vec():
    return TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2), token_pattern=PAT, **LOC)

def clf(C=1.0):
    return LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=42)

vec = make_vec()
Xtr = vec.fit_transform(train["text"])
Xte = vec.transform(test["text"])
model = clf().fit(Xtr, y_train)
pred = model.predict(Xte)
prob = model.predict_proba(Xte)[:, 1]              # modelin "olumlu" için verdiği puan

vs_book("train fingerprint (chapter 4)", fingerprint(train), "8205c2be5ebf")
vs_book("test fingerprint (chapter 4)", fingerprint(test), "634cdb3bb002")
vs_book("features", Xtr.shape[1], 101_659, "{:,}")
base_acc, base_macro, base_f1n = scores(y_test, pred)
vs_book("accuracy", base_acc, 92.57, "{:.2f}")
vs_book("macro-F1", base_macro, 76.21, "{:.2f}")
vs_book("F1 (negative)", base_f1n, 56.47, "{:.2f}")

train fingerprint (chapter 4)            this run 8205c2be5ebf   book 8205c2be5ebf   same as the book
test fingerprint (chapter 4)             this run 634cdb3bb002   book 634cdb3bb002   same as the book
features                                 this run    101,659   book    101,659   same as the book
accuracy                                 this run      92.57   book      92.57   same as the book
macro-F1                                 this run      76.21   book      76.21   same as the book
F1 (negative)                            this run      56.47   book      56.47   same as the book


## 3. Yüz satır okuyun. Kendiniz.

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#sec-rastgele-denetim)

Bölüm, herhangi bir araçtan önce rastgele 100 test satırı çekip okuyor. Çekim, test
kümesinin konumları üzerinde
`np.random.default_rng(42).choice(10_000, 100, replace=False)`; yani aynı yüz satırı siz
de okuyabilirsiniz.

Okuyucunun her satır hakkında verdiği karar bir insan yargısı; kod onu yeniden üretemez.
Bölümün sayımları, tek okuyucuya göre: **0** açık hata (etiket metinle çelişiyor), ikili
bir ürün-hükmü şemasında karara bağlanamaz **7** satır (3'ü "henüz elime ulaşmadı"), iki
yönde de savunulabilir **9** satır, tamam olan **84** satır. Hücre kontrol edebileceğini
kontrol ediyor: çekim bölümün alıntıladığı satırı içeriyor ve 100 satırın 98'i olumlu
etiketli.

**Neye bakacaksınız:** bölümün sayımlarını okumadan önce satırları okuyun ve önce kendi
ölçütlerinizi yazın. Bölümün üçüncü alıştırması tam olarak bunu istiyor.

In [4]:
audit_rows = np.random.default_rng(42).choice(len(test), 100, replace=False)
vs_book("positive reference labels in the sample", int(y_test[audit_rows].sum()), 98)
vs_book("the draw contains test row 4745", 4745 in set(audit_rows.tolist()), True)
print(f"\nrow 4745, label {y_test[4745]}: {test['text'].iloc[4745]!r}\n")

print("the hundred rows   (position in the test set, reference label, text)")
for position in audit_rows:
    print(f"{position:>5}  {y_test[position]}  {test['text'].iloc[position][:150]}")

positive reference labels in the sample  this run         98   book         98   same as the book
the draw contains test row 4745          this run       True   book       True   same as the book

row 4745, label 1: 'sipariş verdim umarım tez ulaşır :)'

the hundred rows   (position in the test set, reference label, text)
 4363  1  öncelikle, ürünün kalitesine söyleyecek hiç birşey yok, gerçekten kaliteli. i̇stediğiniz tüm kahveleri yapması da ayrıcalıklı tarafı. filtre kahve kıs
 9385  1  ürünü salıgünü sipariş verdim cuma günü elimdeydi sorunsuz teslimattan dolayı hb.ye tşk.kokuya gelince güzel koku ama kalıcılıgı daha ön planda kokuyu
 6684  1  1 gün de geldi hızlı teslimat için teşekkürler
 4645  1  kırtasiye marketlerde 2 katından fazlasına satılmakta ..bana beyaz renk olan gönderildi..istediğim de oydu.. enfes.. uzun yıllar kullanılabilecek bir 
 4473  1  süper kalite güzel görünüm. memnun edici.
 7744  1  çocuğuma akşamları okuyorum. beraber hem güzel vakit geçiriyoruz hemde eği

100 satırda sıfır açık hata, hata oranının sıfır olduğu anlamına gelmez. Bölümün
aritmetiği:

- **üç kuralı**: $n$ bağımsız çekimde sıfır olay görüldüğünde $3/n$, oran için yaklaşık
  tek yönlü %95'lik bir üst sınırdır;
- tekdüze bir önselle, 100 çekimde sıfır olaydan sonraki sonsal ortalama $1/(n+2)$;
- %93,7'lik derlem oranına karşı 100 satırda 98 olumlu: %6,3'lük olumsuz oranında 100
  çekimde en çok 2 olumsuz çıkma olasılığı.

Üç kuralı yalnızca sayım sıfır olduğunda geçerli. Kendi örnekleminizde bir ya da daha çok
hata bulursanız onun yerine bir binom aralığı hesaplayın (bölüm Wilson aralığını
kullanıyor).

In [5]:
n = 100
exact_bound = 1 - 0.05 ** (1 / n)                  # (1 - p)^n = 0.05 denkleminin çözümü
vs_book("rule of three, 3/n, %", 100 * 3 / n, 3.0, "{:.1f}")
print(f"  the exact bound it approximates: {100 * exact_bound:.2f}%")
vs_book("posterior mean 1/(n+2), %", 100 / (n + 2), 0.98, "{:.2f}")
vs_book("P(at most 2 negatives in 100), %", 100 * binom.cdf(2, n, 0.063), 4.5, "{:.1f}")


def wilson(k, n, z=1.96):
    """k/n oranı için Wilson skor aralığı."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


print("\nif the count is not zero, e.g. 2 errors in 100:  Wilson 95% interval "
      + "{:.1f}% to {:.1f}%".format(*(100 * x for x in wilson(2, 100))))

rule of three, 3/n, %                    this run        3.0   book        3.0   same as the book
  the exact bound it approximates: 2.95%
posterior mean 1/(n+2), %                this run       0.98   book       0.98   same as the book
P(at most 2 negatives in 100), %         this run        4.5   book        4.5   same as the book

if the count is not zero, e.g. 2 errors in 100:  Wilson 95% interval 0.6% to 7.0%


## 4. Şema da etiketleyiciler kadar zarar verebilir

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#şema-da-etiketleyiciler-kadar-zarar-verebilir)

Bir yorum ürünü övüp teslimattan şikâyet edebilir. Tek bir olumlu/olumsuz bit ikisinden
birini atmak zorunda. Bölüm böyle yorumların *adaylarını* bulmak için, hepsi aşağı yukarı
"ama" anlamına gelen dört Türkçe kelimeyi işaretliyor. Kural bir ön eleme aracı: eşleşen
bir yorumun gerçekten iki karşıt hüküm içerip içermediği okunmadan bilinemez ve kuralın
precision'ı ölçülmedi.

**Neye bakacaksınız:** eşleşen grubun büyüklüğü, olumsuz etiket payı ve modelin iki
gruptaki skorları.

In [6]:
# tr_lower: 4. bölümdeki Türkçeye duyarlı küçük harf dönüştürücü (İ->i, I->ı)
CONTRAST = re.compile(r"\b(ama|fakat|ancak|lakin)\b")
has_but = np.array([bool(CONTRAST.search(tr_lower(t))) for t in test.text])

vs_book("reviews matching the rule", int(has_but.sum()), 2_003, "{:,}")
BOOK_CONTRAST = {True: (20.03, 11.48, 84.07, 72.11, 53.84), False: (79.97, 5.03, 94.70, 77.72, 58.27)}
for group, name in [(True, "rule matches"), (False, "no match")]:
    mask = has_but == group
    acc, macro, f1n = scores(y_test[mask], pred[mask])
    book = BOOK_CONTRAST[group]
    print(f"\n{name}")
    vs_book("  share of test set, %", 100 * mask.mean(), book[0], "{:.2f}")
    vs_book("  reference negative rate, %", 100 * (y_test[mask] == 0).mean(), book[1], "{:.2f}")
    vs_book("  model accuracy", acc, book[2], "{:.2f}")
    vs_book("  model macro-F1", macro, book[3], "{:.2f}")
    vs_book("  model F1 (negative)", f1n, book[4], "{:.2f}")

reviews matching the rule                this run      2,003   book      2,003   same as the book

rule matches
  share of test set, %                   this run      20.03   book      20.03   same as the book
  reference negative rate, %             this run      11.48   book      11.48   same as the book
  model accuracy                         this run      84.07   book      84.07   same as the book
  model macro-F1                         this run      72.11   book      72.11   same as the book
  model F1 (negative)                    this run      53.84   book      53.84   same as the book

no match
  share of test set, %                   this run      79.97   book      79.97   same as the book
  reference negative rate, %             this run       5.03   book       5.03   same as the book
  model accuracy                         this run      94.70   book      94.70   same as the book
  model macro-F1                         this run      77.72   book      77.72   same as the b

**Nasıl okunur.** Test kümesinin beşte biri eşleşiyor, olumsuz oranı iki katından fazla ve
model bu grupta daha düşük skor alıyor. Bölüm bunun neyi gösterdiği ve neyi göstermediği
konusunda dikkatli; yorumlamadan önce üç şeyi ölçüyor:

- macro-F1 farkının ne kadar belirsiz olduğu (test satırları üzerinde bir bootstrap);
- iki grubun başka bir bakımdan ayrışıp ayrışmadığı (ayrışıyorlar: uzunluk);
- kuralın ne kadarının, "yalnızca" ve "güçlükle" anlamına da gelen *ancak* sözcüğüne
  dayandığı.

In [7]:
rng = np.random.default_rng(42)
gaps = []
for _ in range(5_000):
    i = rng.integers(0, len(y_test), len(y_test))
    m = has_but[i]
    gaps.append(macro_f1(y_test[i][~m], pred[i][~m]) - macro_f1(y_test[i][m], pred[i][m]))
observed = macro_f1(y_test[~has_but], pred[~has_but]) - macro_f1(y_test[has_but], pred[has_but])
vs_book("macro-F1 gap, no match − match", observed, 5.6, "{:.1f}")
vs_book("  bootstrap 95%, lower end", np.percentile(gaps, 2.5), 2.5, "{:.1f}")
vs_book("  bootstrap 95%, upper end", np.percentile(gaps, 97.5), 8.8, "{:.1f}")

n_tokens = np.array([len(tokenize(t)) for t in test.text])
vs_book("median tokens, rule matches", int(np.median(n_tokens[has_but])), 30)
vs_book("median tokens, no match", int(np.median(n_tokens[~has_but])), 16)

lowered = [tr_lower(t) for t in test.text]
with_ancak = np.array([bool(re.search(r"\bancak\b", t)) for t in lowered])
others = np.array([bool(re.search(r"\b(ama|fakat|lakin)\b", t)) for t in lowered])
vs_book("reviews containing 'ancak'", int(with_ancak.sum()), 351)
vs_book("  matching through 'ancak' alone", int((with_ancak & ~others).sum()), 287)
print(f"macro-F1 with 'ancak' taken out of the rule: match {macro_f1(y_test[others], pred[others]):.2f}, "
      f"no match {macro_f1(y_test[~others], pred[~others]):.2f}")

macro-F1 gap, no match − match           this run        5.6   book        5.6   same as the book
  bootstrap 95%, lower end               this run        2.5   book        2.5   same as the book
  bootstrap 95%, upper end               this run        8.8   book        8.8   same as the book
median tokens, rule matches              this run         30   book         30   same as the book
median tokens, no match                  this run         16   book         16   same as the book
reviews containing 'ancak'               this run        351   book        351   same as the book
  matching through 'ancak' alone         this run        287   book        287   same as the book
macro-F1 with 'ancak' taken out of the rule: match 71.29, no match 77.80


**Nasıl okunur.** Fark var, aralığı geniş ve *ancak* kuraldan çıkarıldığında da duruyor.
Ama eşleşen yorumlar yaklaşık iki kat uzun ve buradaki hiçbir yorum yön yön yeniden
etiketlenmedi. Dolayısıyla ölçüm, "şema iki hükmü tek bite sıkıştırıyor" ile "model bu
cümleleri kötü okuyor" açıklamaları arasında seçim yapmıyor. Bölümün şema tarafındaki
cevabı, karara bağlanmış vakalar içeren yazılı bir kılavuz; bölümün o kısmı düzyazı ve
kodu yok.

## 5. Uyum: %93 tek başına ne anlatır?

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#uyum-ve-93lük-bir-uyumun-neden-değersiz-olabileceği)

Etiketlere güvenilip güvenilemeyeceğini bilmek için aynı öğeler hakkında ikinci bir görüşe
ve ikisinin ne sıklıkta örtüştüğünü veren bir sayıya ihtiyacınız var. Bölümün ikinci bir
insan etiketleyicisi yoktu; bu yüzden referans etiketleri (A), 10.000 test satırında bir
sözlük kuralıyla (B) karşılaştırıyor. Bunun ne olduğunu açıkça söylüyor: referans
etiketler ile, etiketleri görmüş biri tarafından yazılmış bir kural arasındaki örtüşme.
Aritmetiği gösteriyor; bir insan uyumu çalışması değil.

Kuralın ifade listeleri bölümde basılmıyor, dolayısıyla tablosu burada veri olarak
alınıyor. Tablodan türeyen her şey yeniden hesaplanıyor.

**Neye bakacaksınız:** ham uyum %93,42; A ile B kendi oranlarıyla bağımsız etiketleseydi
beklenecek uyum ise zaten %90,99.

In [8]:
# bölümün tablosu: A (referans, satırlar) ve B (sözlük kuralı, sütunlar), önce olumsuz
BOOK_TABLE = np.array([[141, 491],
                       [167, 9201]], dtype=float)


def coefficients(table):
    """İki değerlendirici ve iki kategori için uyum katsayıları, 2x2 bir tablodan."""
    table = np.asarray(table, dtype=float)
    n = table.sum()
    p_o = np.trace(table) / n
    a_rates, b_rates = table.sum(axis=1) / n, table.sum(axis=0) / n
    p_e = float(a_rates @ b_rates)                              # her değerlendirici kendi oranını korur
    kappa = (p_o - p_e) / (1 - p_e)
    # Krippendorff alpha'sı, nominal: gözlenen uyuşmazlık / havuzlanmış etiketlerden beklenen
    pooled = table + table.T
    totals, N = pooled.sum(axis=1), pooled.sum()
    alpha = 1 - ((N - np.trace(pooled)) / N) / ((N * N - (totals ** 2).sum()) / (N * (N - 1)))
    mean_rates = (a_rates + b_rates) / 2
    chance_ac1 = 2 * mean_rates[0] * mean_rates[1]              # Gwet AC1, iki kategori
    return {"p_o": p_o, "p_e": p_e, "kappa": kappa, "alpha": alpha,
            "AC1": (p_o - chance_ac1) / (1 - chance_ac1), "PABAK": 2 * p_o - 1,
            "A positive": a_rates[1], "B positive": b_rates[1]}


book = coefficients(BOOK_TABLE)
vs_book("A calls positive, %", 100 * book["A positive"], 93.68, "{:.2f}")
vs_book("B calls positive, %", 100 * book["B positive"], 96.92, "{:.2f}")
vs_book("raw agreement p_o", book["p_o"], 0.9342, "{:.4f}")
vs_book("expected agreement p_e", book["p_e"], 0.9099, "{:.4f}")
print("\n### the identical 10,000-row table, five coefficients")
vs_book("raw agreement  p_o", book["p_o"], 0.9342, "{:.4f}")
vs_book("Gwet's AC1", book["AC1"], 0.9277, "{:.4f}")
vs_book("PABAK  (2*p_o - 1)", book["PABAK"], 0.8684, "{:.4f}")
vs_book("Cohen's kappa", book["kappa"], 0.2698, "{:.4f}")
vs_book("Krippendorff's alpha", book["alpha"], 0.2655, "{:.4f}")

A calls positive, %                      this run      93.68   book      93.68   same as the book
B calls positive, %                      this run      96.92   book      96.92   same as the book
raw agreement p_o                        this run     0.9342   book     0.9342   same as the book
expected agreement p_e                   this run     0.9099   book     0.9099   same as the book

### the identical 10,000-row table, five coefficients
raw agreement  p_o                       this run     0.9342   book     0.9342   same as the book
Gwet's AC1                               this run     0.9277   book     0.9277   same as the book
PABAK  (2*p_o - 1)                       this run     0.8684   book     0.8684   same as the book
Cohen's kappa                            this run     0.2698   book     0.2698   same as the book
Krippendorff's alpha                     this run     0.2655   book     0.2655   same as the book


**Nasıl okunur.** Beş katsayı, tek tablo, 0,27 ile 0,93 arasında değerler. Şans hakkında
farklı sorulara cevap veriyorlar: kappa her değerlendiricinin kendi oranını korumasına
izin veriyor, alpha iki değerlendiricinin etiketlerini havuzluyor, AC1 ve PABAK ise şansla
çok daha az uyum varsayıyor. Bunlardan birini tablo ve iki değerlendiricinin oranları
olmadan bildirmek, hangi sorunun sorulduğunu gizler.

Bu tabloda kappa'nın alpha'ya yakın olması bağımsız bir doğrulama değil. İkisi de aynı
etiket çiftlerinden hesaplanıyor.

### Gerisi çalıştırılabilsin diye kendi kuralımız

Bölümün sonraki adımı her satır için B'nin etiketini gerektiriyor; tablo bunu içermiyor.
Bu yüzden burada küçük ve şeffaf bir kural var: 12 olumsuz ve 10 olumlu ifade; 1. bölümün
notebook'unun kullandığı kural. Biçimi bölümünkiyle aynı (yalnızca olumsuz bir ifade
eşleşip hiçbir olumlu ifade eşleşmediğinde olumsuz). **Sayıları bölümün sayıları değil.**

In [9]:
NEG = ["kötü", "berbat", "rezalet", "değil", "beğenmedim", "memnun kalmadım",
       "tavsiye etmem", "tavsiye etmiyorum", "iade", "bozuk", "kalitesiz", "pişman"]
POS = ["güzel", "harika", "mükemmel", "süper", "memnunum", "teşekkür",
       "tavsiye ederim", "beğendim", "kaliteli", "başarılı"]


def annotator_b(text):
    t = tr_lower(text).replace("̇", "")       # 4. bölüm: başıboş birleştirici nokta
    neg = any(p in t for p in NEG)
    pos = any(p in t for p in POS)
    if neg and not pos:
        return 0
    return 1          # eşleşme yoksa ya da iki tür eşleşme de varsa: olumlu


def table_of(a, b):
    return np.array([[((a == i) & (b == j)).sum() for j in (0, 1)] for i in (0, 1)], dtype=float)


b_ours = np.array([annotator_b(t) for t in test["text"]])
ours_table = table_of(y_test, b_ours)
print("A (reference, rows) vs our 22-phrase rule (columns), n = 10,000\n")
print("                B: negative   B: positive")
for i, name in enumerate(["A: negative", "A: positive"]):
    print(f"{name}     {int(ours_table[i, 0]):>9,}     {int(ours_table[i, 1]):>9,}")
ours = coefficients(ours_table)
print(f"\nraw agreement {100 * ours['p_o']:.2f}%   p_e {100 * ours['p_e']:.2f}%   "
      f"kappa {ours['kappa']:.4f}   alpha {ours['alpha']:.4f}   (chapter's rule: 93.42%, 90.99%, 0.2698, 0.2655)")
print(f"scikit-learn's cohen_kappa_score on the same labels: {cohen_kappa_score(y_test, b_ours):.4f}")

A (reference, rows) vs our 22-phrase rule (columns), n = 10,000



                B: negative   B: positive
A: negative           200           432
A: positive           303         9,065

raw agreement 92.65%   p_e 89.29%   kappa 0.3140   alpha 0.3135   (chapter's rule: 93.42%, 90.99%, 0.2698, 0.2655)
scikit-learn's cohen_kappa_score on the same labels: 0.3140


### Aynı referans ve kural, farklı örneklem

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#aynı-etiketleyiciler-farklı-bir-sayı)

A'nın olumsuz dediği 632 yorumun tamamını ve A'nın olumlu dediklerinden rastgele 632
tanesini alın. Aynı iki etiket kaynağı, sınıf oranı farklı bir örneklemde. Bölümün *kendi*
kuralı için tablosu:

```text
                    full test set (n=10,000)   balanced (n=1,264)
raw agreement                       93.42%              60.28%
p_e                                 90.99%              50.00%
Cohen's kappa                        0.2698              0.2057
Krippendorff's alpha                 0.2655              0.0722
```

Aşağıdaki hücre aynısını bizim kuralımızla yapıyor. Sayılar bölümünkilerden farklı;
karşılaştırılacak olan örüntü.

**Neye bakacaksınız:** ham uyum onlarca puan düşüyor, $p_e$ tam 0,5 oluyor ve tam kümede
uyuşan kappa ile alpha birbirinden ayrılıyor.

In [10]:
negatives = np.where(y_test == 0)[0]
positives = np.random.default_rng(42).choice(np.where(y_test == 1)[0], len(negatives), replace=False)
balanced = np.concatenate([negatives, positives])
bal = coefficients(table_of(y_test[balanced], b_ours[balanced]))

print(f"### our 22-phrase rule\n{'':<24}{'full test set (n=10,000)':>26}{'balanced (n=' + format(len(balanced), ',') + ')':>22}")
for label, key, fmt in [("raw agreement", "p_o", "{:.2%}"), ("p_e", "p_e", "{:.2%}"),
                        ("Cohen's kappa", "kappa", "{:.4f}"), ("Krippendorff's alpha", "alpha", "{:.4f}")]:
    print(f"{label:<24}{fmt.format(ours[key]):>26}{fmt.format(bal[key]):>22}")

### our 22-phrase rule
                          full test set (n=10,000)    balanced (n=1,264)
raw agreement                               92.65%                64.56%
p_e                                         89.29%                50.00%
Cohen's kappa                               0.3140                0.2911
Krippendorff's alpha                        0.3135                0.2053


**Nasıl okunur.** Dengelenmiş altörneklemde $p_e$, kural ne yaparsa yapsın tam 0,5: bir
değerlendirici 50/50 bölündüğünde beklenen uyum, diğerinin $b$ oranı ne olursa olsun
$0.5\,b + 0.5\,(1 - b) = 0.5$. Alpha iki değerlendiricinin etiketlerini havuzluyor ve
kuralı, bu örneklemin yarısından çok fazlasına olumlu dediği için cezalandırıyor; kappa
cezalandırmıyor. Sütunların hiçbiri bu iki kaynağın "asıl" uyumu değil. Her biri kendi
örnekleminin sınıf oranına ait; bölümün her katsayının yanında tabloyu ve marjinalleri
istemesinin sebebi bu.

### Bir kappa'nın çevresindeki aralık ne kadar geniş?

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#uzlaştırma)

Bir pilot etiketlemede 10.000 değil, birkaç yüz öğe olur. Bölüm, 0,27'lik bir kappa'nın o
boyutta neye benzediğini görmek için kendi tablosundan yeniden örnekliyor: aynı öğenin iki
yargısı birlikte çekiliyor.

**Neye bakacaksınız:** rastgele çekilmiş 200 öğedeki standart hata, ve referans etiketi
olumsuz 100 ile olumlu 100 öğe olarak tabakalanmış 200 öğedeki standart hata.

In [11]:
rng = np.random.default_rng(42)
cell_probabilities = (BOOK_TABLE / BOOK_TABLE.sum()).ravel()


def kappa_draws(n_items, n_boot=2_000):
    """Bölümün tablosundan çekilen n_items öğelik (yargı çiftleri) n_boot örneklemin kappa'sı."""
    draws = rng.multinomial(n_items, cell_probabilities, size=n_boot).reshape(n_boot, 2, 2)
    return np.array([coefficients(t)["kappa"] for t in draws])


def kappa_se_stratified(n_per_class=100, n_boot=2_000):
    """Aynısı, referans etiketi olumsuz n ve olumlu n öğeden oluşan bir pilot için."""
    neg_hit = rng.binomial(n_per_class, BOOK_TABLE[0, 0] / BOOK_TABLE[0].sum(), n_boot)
    pos_hit = rng.binomial(n_per_class, BOOK_TABLE[1, 1] / BOOK_TABLE[1].sum(), n_boot)
    return np.nanstd([coefficients([[a, n_per_class - a], [n_per_class - d, d]])["kappa"]
                      for a, d in zip(neg_hit, pos_hit)])


draws_200 = kappa_draws(200)
vs_book("standard error, 10,000 items", np.nanstd(kappa_draws(10_000)), 0.02, "{:.2f}")
vs_book("standard error, 200 random items", np.nanstd(draws_200), 0.14, "{:.2f}")
vs_book("standard error, 100 + 100 stratified", kappa_se_stratified(), 0.04, "{:.2f}")
low, high = np.nanpercentile(draws_200, [2.5, 97.5])
print(f"\n200 random items: the middle 95% of the resampled kappas runs from {low:.2f} to {high:.2f}"
      "   (book: roughly [−0.02, 0.54])")

standard error, 10,000 items             this run       0.02   book       0.02   same as the book
standard error, 200 random items         this run       0.14   book       0.14   same as the book
standard error, 100 + 100 stratified     this run       0.04   book       0.04   same as the book

200 random items: the middle 95% of the resampled kappas runs from -0.03 to 0.53   (book: roughly [−0.02, 0.54])


Bölümün kendi bootstrap kodu iki etiket vektörü üzerinde çalışıyor; bir pilottan sonra
elinizde olacak olan da bu. Aşağıda bizim kuralımız ve tam test kümesi üzerinde.

In [12]:
a, b = y_test, b_ours
# bölümün bootstrap'i
rng = np.random.default_rng(42)
idx = (rng.integers(0, len(a), len(a)) for _ in range(2000))
boot = [cohen_kappa_score(a[i], b[i]) for i in idx]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"our rule, n = {len(a):,}: kappa {cohen_kappa_score(a, b):.4f}, bootstrap 95% [{lo:.4f}, {hi:.4f}]")

our rule, n = 10,000: kappa 0.3140, bootstrap 95% [0.2768, 0.3503]


**Nasıl okunur.** Bu kadar çarpık bir derlemden rastgele çekilmiş 200 öğede 0,27'nin
çevresindeki aralık aşağı yukarı sıfırdan 0,54'e uzanıyor. Tabakalı bir pilot aralığı çok
daraltıyor, ama bir bedeli var: kappa'sı doğal orana değil, seçtiğiniz 50/50 oranına ait.

## 6. Etiketleri ölçekte denetlemek

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#sec-etiket-denetimi)

40.000 satırı kimse okuyamaz. Bölüm, hangi satırların önce okunacağına karar vermek için
modeli kullanıyor: eğitim kümesinin beşte dördünde eğit, kalan beşte biri tahmin et,
döndür ve modelin, etiketin taşı*madığı* sınıfa yüksek puan verdiği satırları işaretle.
Bu, confident learning'den esinlenen basit bir eşik kuralı ve bölüm onun ne olmadığını
söylüyor: puan, etiketin yanlış olma olasılığı değil.

Kod bölümün. Yorumu önemli: buradaki vektörleştirici katlardan önce, 40.000 satırın
tamamında eğitildi; dolayısıyla tahminler tam anlamıyla out-of-fold değil. Bundan
sonraki hücre, vektörleştirici her katın içinde eğitildiğinde neyin değiştiğini ölçüyor.

**Neye bakacaksınız:** yüksek bir eşiği ne kadar az satırın geçtiği ve bunların iki sınıf
arasında nasıl bölündüğü.

In [13]:
y = y_train
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Xtr burada, katlardan önce 40.000 satırın tamamında eğitilmiş bir vektörleştiriciden geliyor.
# Tam out-of-fold tahmin için vektörleştiriciyi de her eğitim katında eğitin.
oof = np.zeros(len(y))
for train_idx, val_idx in skf.split(Xtr, y):
    oof[val_idx] = clf().fit(Xtr[train_idx], y[train_idx]).predict_proba(Xtr[val_idx])[:, 1]

# modelin karşı sınıfa verdiği puan; etiketin yanlış olma olasılığı değildir
wrong_conf = np.where(y == 1, 1 - oof, oof)

BOOK_FLAGS = {0.50: (3_026, 2_232, 794), 0.90: (92, 49, 43), 0.95: (34, 18, 16), 0.99: (2, 2, 0)}
for threshold, (b_total, b_pos, b_neg) in BOOK_FLAGS.items():
    flagged = wrong_conf > threshold
    print(f"> {threshold:.2f}")
    vs_book("  rows flagged", int(flagged.sum()), b_total, "{:,}")
    vs_book("  reference positive", int((flagged & (y == 1)).sum()), b_pos, "{:,}")
    vs_book("  reference negative", int((flagged & (y == 0)).sum()), b_neg, "{:,}")

flag = wrong_conf > 0.90
print()
vs_book("flag rate of negative labels, %", 100 * (flag & (y == 0)).sum() / (y == 0).sum(), 1.70, "{:.2f}")
vs_book("flag rate of positive labels, %", 100 * (flag & (y == 1)).sum() / (y == 1).sum(), 0.13, "{:.2f}")

> 0.50
  rows flagged                           this run      3,026   book      3,026   same as the book
  reference positive                     this run      2,232   book      2,232   same as the book
  reference negative                     this run        794   book        794   same as the book
> 0.90
  rows flagged                           this run         92   book         92   same as the book
  reference positive                     this run         49   book         49   same as the book
  reference negative                     this run         43   book         43   same as the book
> 0.95
  rows flagged                           this run         34   book         34   same as the book
  reference positive                     this run         18   book         18   same as the book
  reference negative                     this run         16   book         16   same as the book
> 0.99
  rows flagged                           this run          2   book          2   same as t

**Nasıl okunur.** %6,3'ü olumsuz olan bir derlemde işaretler sınıflar arasında neredeyse
dengeli: olumsuz bir etiket, olumlu bir etiketin on üç katı sıklıkta işaretleniyor. Bu
hata oranlarındaki değil, *işaretlenme* oranlarındaki bir fark. Model azınlık sınıfında
yalnızca daha az emin ya da daha kötü kalibre edilmiş olabilir.

Şimdi tam anlamıyla out-of-fold sürüm: vektörleştirici yalnızca her katın eğitim kısmında
eğitiliyor.

In [14]:
oof_infold = np.zeros(len(y))
for train_idx, val_idx in skf.split(Xtr, y):
    fold_vec = make_vec()
    X_fit = fold_vec.fit_transform(train["text"].iloc[train_idx])
    X_val = fold_vec.transform(train["text"].iloc[val_idx])
    oof_infold[val_idx] = clf().fit(X_fit, y[train_idx]).predict_proba(X_val)[:, 1]
wrong_conf_infold = np.where(y == 1, 1 - oof_infold, oof_infold)

vs_book("flags above 0.90, vectoriser in the fold", int((wrong_conf_infold > 0.90).sum()), 123)
vs_book("flags above 0.95, vectoriser in the fold", int((wrong_conf_infold > 0.95).sum()), 47)
vs_book("of the first 92, also in the new list", int((flag & (wrong_conf_infold > 0.90)).sum()), 89)

flags above 0.90, vectoriser in the fold this run        123   book        123   same as the book
flags above 0.95, vectoriser in the fold this run         47   book         47   same as the book
of the first 92, also in the new list    this run         89   book         89   same as the book


**Nasıl okunur.** Vektörleştiriciyi katların dışında eğitmek, her katın puanlayacağı
satırların sözlüğünü ve belge frekanslarını görmesine izin veriyor; model de olması
gerekenden biraz daha emin görünüyor. Bu düzeltilince liste uzuyor ve ilk listenin
neredeyse tamamı yerinde kalıyor. Kendi denetiminizde vektörleştiriciyi ve
sınıflandırıcıyı tek bir `Pipeline` içine koyun; bunu kendiliğinden yapar. Aşağıdaki
sayılar, bölümdeki gibi, 92 satırlık ilk listeye ait.

### İşaret bir okuma çağrısıdır

92 aday eğitim kümesinde. Bölümün yazarı ayrı bir denetimde, modelin saklanan etikete en
yüksek puanla itiraz ettiği 40 *test* yorumunu okudu ve elle sınıflandırdı: **11** `error`
(etiket metinle açıkça çelişiyor), **20** `schema` (tek bir ikili etiket metni temsil
edemiyor), **9** `model` (etiket savunulabilir, sınıflandırıcı yanılıyor). Bunlar, modelin
seçtiği bir örneklem üzerinde, kör olmayan tek okuyucunun yargıları. Bölüm bunların derlem
ya da 92 eğitim adayı için bir kestirim olmadığını ve sayı alıntılanmadan önce ikinci, kör
bir okuyucunun okumayı tekrarlaması gerektiğini söylüyor.

Hücre aynı kırk satırı okumanız için basıyor, bölümün tartıştığı dördünü kontrol ediyor ve
bölümün iki pay için verdiği Wilson aralıklarını hesaplıyor.

In [15]:
test_wrong_conf = np.where(y_test == 1, 1 - prob, prob)
top40 = np.argsort(-test_wrong_conf)[:40]

for row, book_label, book_p in [(6874, 0, 0.984), (8356, 1, 0.059), (8006, 1, 0.034), (2544, 1, None)]:
    print(f"row {row}: in the forty: {row in set(top40.tolist())}")
    vs_book("  reference label", int(y_test[row]), book_label)
    if book_p is not None:
        vs_book("  model p(positive)", prob[row], book_p, "{:.3f}")
print()
for label, k, book_low, book_high in [("schema share 20/40", 20, 35, 65), ("error share 11/40", 11, 16, 43)]:
    low, high = wilson(k, 40)
    vs_book(f"{label}: Wilson lower, %", 100 * low, book_low, "{:.0f}")
    vs_book(f"{label}: Wilson upper, %", 100 * high, book_high, "{:.0f}")

print("\nthe forty rows   (position, reference label, model p(positive), text)")
for row in top40:
    print(f"{row:>5}  {y_test[row]}  {prob[row]:.3f}  {test['text'].iloc[row][:140]}")

row 6874: in the forty: True
  reference label                        this run          0   book          0   same as the book
  model p(positive)                      this run      0.984   book      0.984   same as the book
row 8356: in the forty: True
  reference label                        this run          1   book          1   same as the book
  model p(positive)                      this run      0.059   book      0.059   same as the book
row 8006: in the forty: True
  reference label                        this run          1   book          1   same as the book
  model p(positive)                      this run      0.034   book      0.034   same as the book
row 2544: in the forty: True
  reference label                        this run          1   book          1   same as the book

schema share 20/40: Wilson lower, %      this run         35   book         35   same as the book
schema share 20/40: Wilson upper, %      this run         65   book         65   same as the book
e

Bu satırların ikisinde bölüm, modeli hangi özniteliklerin oynattığına bakıyor. Doğrusal
bir modelde her özniteliğin karar puanına katkısı, TF-IDF değeri çarpı katsayısıdır.

**Neye bakacaksınız:** 8006 numaralı satırda `iade` kelimesi baskın. 2544 numaralı satırda
`fena değil` bigram'ı sözlükte ama neredeyse hiç katkı vermiyor; kararı `değil` ve `fena`
unigram'ları veriyor.

In [16]:
feature_names = np.array(vec.get_feature_names_out())
coefficients_ = model.coef_[0]


def contributions(row):
    x = Xte[row]
    values = x.multiply(coefficients_).toarray()[0]
    return {feature_names[j]: values[j] for j in x.indices}


c = contributions(8006)
print(f"row 8006: {test['text'].iloc[8006][:110]}")
for feature, book_value in [("iade", -1.78), ("beğenmedim", -1.30), ("iade ettim", -0.65)]:
    vs_book(f"  contribution of '{feature}'", c[feature], book_value, "{:+.2f}")
negative_total = sum(v for v in c.values() if v < 0)
iade_share = sum(v for f, v in c.items() if "iade" in f and v < 0) / negative_total
print(f"  share of the negative contributions from features containing 'iade': {100 * iade_share:.0f}%   (book: about half)")

c = contributions(2544)
print(f"\nrow 2544: {test['text'].iloc[2544]}")
for feature, book_value in [("değil", -1.77), ("fena", -0.55), ("fena değil", -0.06)]:
    vs_book(f"  contribution of '{feature}'", c[feature], book_value, "{:+.2f}")

row 8006: hiç bir sebeb göstermeden iade kurallarına uyarak beğenmedim dedim, iade ettim.ücret iadem anında yapıldı.karg
  contribution of 'iade'                 this run      -1.78   book      -1.78   same as the book
  contribution of 'beğenmedim'           this run      -1.30   book      -1.30   same as the book
  contribution of 'iade ettim'           this run      -0.65   book      -0.65   same as the book
  share of the negative contributions from features containing 'iade': 51%   (book: about half)

row 2544: paketleme başarısız. ampüller fena değil
  contribution of 'değil'                this run      -1.77   book      -1.77   same as the book
  contribution of 'fena'                 this run      -0.55   book      -0.55   same as the book
  contribution of 'fena değil'           this run      -0.06   book      -0.06   same as the book


### Temizlik gerçekte ne kazandırdı

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#temizlik-gerçekte-ne-kazandırdı)

Elde 92 şüpheli satır varken akla gelen hamle onları silmek ya da etiketlerini çevirip
yeniden eğitmek. Bölüm ikisini de üç eşikte yapıyor ve aynı test bölmesinde puanlıyor.
Bunu keşif amaçlı bir karşılaştırma diye adlandırıyor; hiç görülmemiş bir test kümesindeki
bir sonuç değil.

**Neye bakacaksınız:** tablonun hiçbir satırı baseline'ın üstünde değil.

In [17]:
def retrain(keep=None, labels=None):
    labels = y_train if labels is None else labels
    keep = np.ones(len(y_train), bool) if keep is None else keep
    return scores(y_test, clf().fit(Xtr[keep], labels[keep]).predict(Xte))


BOOK_REPAIR = {("drop", 0.90): (92.37, 75.72, 55.61), ("flip", 0.90): (92.47, 75.94, 55.99),
               ("drop", 0.95): (92.39, 75.83, 55.83), ("flip", 0.95): (92.52, 76.03, 56.15),
               ("drop", 0.99): (92.54, 76.15, 56.37), ("flip", 0.99): (92.52, 76.11, 56.31)}
repaired = {}
print("### repairs, judged on the test split used throughout the chapter")
print(f"{'':<24}{'acc':>7}{'macro-F1':>11}{'F1(neg)':>10}   book macro-F1")
print(f"{'baseline (no repair)':<24}{base_acc:>7.2f}{base_macro:>11.2f}{base_f1n:>10.2f}")
for threshold in (0.90, 0.95, 0.99):
    flagged = wrong_conf > threshold
    flipped = y_train.copy()
    flipped[flagged] = 1 - flipped[flagged]
    for action, result in [("drop", retrain(keep=~flagged)), ("flip", retrain(labels=flipped))]:
        repaired[(action, threshold)] = result
        book = BOOK_REPAIR[(action, threshold)]
        same = "same" if all(f"{r:.2f}" == f"{b:.2f}" for r, b in zip(result, book)) else "DIFFERS"
        print(f"{action} {int(flagged.sum()):>2}  (p > {threshold:.2f})      {result[0]:>7.2f}{result[1]:>11.2f}{result[2]:>10.2f}   {book[1]:.2f}  {same}")

### repairs, judged on the test split used throughout the chapter
                            acc   macro-F1   F1(neg)   book macro-F1
baseline (no repair)      92.57      76.21     56.47


drop 92  (p > 0.90)        92.37      75.72     55.61   75.72  same
flip 92  (p > 0.90)        92.47      75.94     55.99   75.94  same


drop 34  (p > 0.95)        92.39      75.83     55.83   75.83  same
flip 34  (p > 0.95)        92.52      76.03     56.15   76.03  same
drop  2  (p > 0.99)        92.54      76.15     56.37   76.15  same
flip  2  (p > 0.99)        92.52      76.11     56.31   76.11  same


Böyle bir tablonun bir kontrole ihtiyacı var. *Herhangi* 92 satırı silmek ya da çevirmek
skoru aşağı yukarı aynı kadar oynatıyorsa, işaretlenen satırlar özel değildir. Bölüm,
işaretlerin sınıf karışımıyla (49 olumlu, 43 olumsuz) rastgele 92 satırı yirmi kez çekiyor
ve aynı iki işlemi uyguluyor.

In [18]:
n_pos, n_neg = int((flag & (y_train == 1)).sum()), int((flag & (y_train == 0)).sum())
positive_rows, negative_rows = np.where(y_train == 1)[0], np.where(y_train == 0)[0]
rng = np.random.default_rng(0)
dropped, flipped_scores = [], []
for _ in range(20):
    pick = np.concatenate([rng.choice(positive_rows, n_pos, replace=False),
                           rng.choice(negative_rows, n_neg, replace=False)])
    keep = np.ones(len(y_train), bool)
    keep[pick] = False
    dropped.append(retrain(keep=keep)[1])
    labels = y_train.copy()
    labels[pick] = 1 - labels[pick]
    flipped_scores.append(retrain(labels=labels)[1])

print(f"### control: {n_pos + n_neg} random rows with the class mix of the flags ({n_pos} pos / {n_neg} neg), 20 draws")
for name, values, book in [("drop 92 random rows", dropped, (76.15, 0.13, 75.91, 76.45)),
                           ("flip 92 random rows", flipped_scores, (75.87, 0.36, 75.01, 76.47))]:
    vs_book(f"{name}: mean", np.mean(values), book[0], "{:.2f}")
    vs_book(f"{name}: sd", np.std(values, ddof=1), book[1], "{:.2f}")
    vs_book(f"{name}: min", min(values), book[2], "{:.2f}")
    vs_book(f"{name}: max", max(values), book[3], "{:.2f}")
print(f"\ndrop the 92 flagged rows   {repaired[('drop', 0.90)][1]:.2f}")
print(f"flip the 92 flagged rows   {repaired[('flip', 0.90)][1]:.2f}")

### control: 92 random rows with the class mix of the flags (49 pos / 43 neg), 20 draws
drop 92 random rows: mean                this run      76.15   book      76.15   same as the book
drop 92 random rows: sd                  this run       0.13   book       0.13   same as the book
drop 92 random rows: min                 this run      75.91   book      75.91   same as the book
drop 92 random rows: max                 this run      76.45   book      76.45   same as the book
flip 92 random rows: mean                this run      75.87   book      75.87   same as the book
flip 92 random rows: sd                  this run       0.36   book       0.36   same as the book
flip 92 random rows: min                 this run      75.01   book      75.01   same as the book
flip 92 random rows: max                 this run      76.47   book      76.47   same as the book

drop the 92 flagged rows   75.72
flip the 92 flagged rows   75.94


**Nasıl okunur.** İşaretlenen 92 satırı silmek, yirmi rastgele silmenin en düşüğünün de
altında skor veriyor; onları çevirmek ise rastgele 92 etiketi çevirmekten ayırt
edilemiyor. İşaretlenen satırlar rastgele satırlardan farklı davranıyor, ama skoru
yükselterek değil. Okunmaya değer bir liste üretmek ile o listeyi topluca düzenleyerek
modeli iyileştirmek ayrı hedefler. İkincisine burada ulaşılmadı. 92 satırı silinmeden ya da
çevrilmeden önce kimse okumadı; bölümün tavsiyesi önce okumak.

## 7. Etiket gürültüsünün bedeli ne ve hangi yönde

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#etiket-gürültüsünün-bedeli-ne-ve-hangi-yönde)

Şimdi etiketler bilerek bozuluyor. Eğitim etiketleri kontrollü bir oranda çevriliyor, model
yeniden eğitiliyor ve özgün test etiketleriyle puanlanıyor. Üç rejim var: `symmetric`
verilen payı *her* sınıftan çeviriyor, `neg -> pos` o payı yalnızca olumsuz etiketlerden,
`pos -> neg` ise yalnızca olumlu etiketlerden.

Oran, *kaynak sınıfın* bir payı ve iki sınıfın büyüklüğü çok farklı: olumsuzların %10'u
253 etiket, olumluların %10'u 3.747 etiket.

**Seed'ler.** Bölüm üç seed'in ortalamasını bildiriyor ve hangileri olduğunu basmıyor. Bu
hücre 0, 1 ve 2 seed'lerini kullanıyor ve bölümün değerinden farkı basıyor. Bölüm, kendi
seed'leri üzerindeki standart sapmanın hiçbir zaman 0,50'yi geçmediğini bildiriyor.

Hücre modeli 63 kez eğitiyor ve yaklaşık on saniye sürüyor.

In [19]:
RATES = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.40]
NOISE_SEEDS = [0, 1, 2]
BOOK_NOISE = {"symmetric": [76.21, 74.18, 72.43, 68.58, 63.34, 55.43, 49.86, 43.71],
              "neg -> pos": [76.21, 76.14, 76.29, 76.27, 76.42, 76.85, 77.12, 76.47],
              "pos -> neg": [76.21, 74.17, 72.87, 68.63, 63.63, 56.69, 52.01, 47.99]}


def corrupt(labels, rate, regime, rng):
    """Kaynak sınıf(lar)ın `rate` kadarını çevirir: bütün satırların değil, sınıfın bir payı."""
    noisy = labels.copy()
    sources = {"symmetric": (0, 1), "neg -> pos": (0,), "pos -> neg": (1,)}[regime]
    for source in sources:
        rows = np.where(labels == source)[0]
        chosen = rng.choice(rows, int(round(rate * len(rows))), replace=False)
        noisy[chosen] = 1 - noisy[chosen]
    return noisy


def predict_with(labels):
    return clf().fit(Xtr, labels).predict(Xte)


start = time.perf_counter()
noise_runs = {}
for regime in BOOK_NOISE:
    for rate in RATES:
        if rate == 0:
            noise_runs[(regime, rate)] = [pred]
        else:
            noise_runs[(regime, rate)] = [predict_with(corrupt(y_train, rate, regime, np.random.default_rng(seed)))
                                          for seed in NOISE_SEEDS]
print(f"### macro-F1, original test labels, model retrained at each rate; seeds {NOISE_SEEDS}   ({time.perf_counter() - start:.0f}s)")
print("flip rate   " + "".join(f"{rate:>7.0%}" for rate in RATES))
for regime in BOOK_NOISE:
    means = [np.mean([macro_f1(y_test, p) for p in noise_runs[(regime, rate)]]) for rate in RATES]
    print(f"{regime:<12}" + "".join(f"{m:>7.2f}" for m in means))
    print(f"{'  book':<12}" + "".join(f"{b:>7.2f}" for b in BOOK_NOISE[regime]))
    print(f"{'  Δ':<12}" + "".join(f"{m - b:>+7.2f}" for m, b in zip(means, BOOK_NOISE[regime])))
spread = max(np.std([macro_f1(y_test, p) for p in runs], ddof=1) for runs in noise_runs.values() if len(runs) > 1)
print(f"\nlargest standard deviation over seeds in this run: {spread:.2f}   (book: never above 0.50)")

### macro-F1, original test labels, model retrained at each rate; seeds [0, 1, 2]   (11s)
flip rate        0%     1%     2%     5%    10%    20%    30%    40%
symmetric     76.21  74.51  72.63  68.40  63.08  55.44  49.88  43.51
  book        76.21  74.18  72.43  68.58  63.34  55.43  49.86  43.71
  Δ           -0.00  +0.33  +0.20  -0.18  -0.26  +0.01  +0.02  -0.20
neg -> pos    76.21  76.23  76.17  76.33  76.58  76.67  77.11  77.11
  book        76.21  76.14  76.29  76.27  76.42  76.85  77.12  76.47
  Δ           -0.00  +0.09  -0.12  +0.06  +0.16  -0.18  -0.01  +0.64
pos -> neg    76.21  74.38  72.67  68.31  63.13  56.61  51.44  47.29
  book        76.21  74.17  72.87  68.63  63.63  56.69  52.01  47.99
  Δ           -0.00  +0.21  -0.20  -0.32  -0.50  -0.08  -0.57  -0.70

largest standard deviation over seeds in this run: 0.62   (book: never above 0.50)


**Nasıl okunur.** İki satır dik düşüyor, biri neredeyse hiç oynamıyor. Bu modelle ve bu
sınıf oranıyla olumsuz etiketlerin %40'ını bile olumluya çevirmek macro-F1'i baseline'a
yakın bırakıyor; olumlu etiketlerin %10'unu olumsuza çevirmek ise yaklaşık on iki puana
mal oluyor. Aynı *oran*, bir yönde diğerinin on beş katı satır demek.

Bölüm, ortadaki satırdaki küçük yükselişin nereden geldiğine bakıyor: olumsuzların %30'u
çevrildiğinde model daha seyrek "olumsuz" diyor; dolayısıyla olumsuz sınıfta precision
yükseliyor, recall düşüyor.

In [20]:
runs = noise_runs[("neg -> pos", 0.30)]
print(f"{'':<22}{'predicted negative':>20}{'precision (neg)':>18}{'recall (neg)':>15}")
print(f"{'baseline':<22}{int((pred == 0).sum()):>20,}{100 * precision_score(y_test, pred, pos_label=0):>17.1f}%"
      f"{100 * recall_score(y_test, pred, pos_label=0):>14.1f}%")
print(f"{'30% neg -> pos (mean)':<22}{np.mean([(p == 0).sum() for p in runs]):>20,.0f}"
      f"{100 * np.mean([precision_score(y_test, p, pos_label=0) for p in runs]):>17.1f}%"
      f"{100 * np.mean([recall_score(y_test, p, pos_label=0) for p in runs]):>14.1f}%")
print("book: about 845 instead of 1,075 predicted negative; precision 44.8% -> about 50%; recall 76.3% -> about 67%\n")
vs_book("baseline: predicted negative", int((pred == 0).sum()), 1_075, "{:,}")
vs_book("baseline: precision (neg), %", 100 * precision_score(y_test, pred, pos_label=0), 44.8, "{:.1f}")
vs_book("baseline: recall (neg), %", 100 * recall_score(y_test, pred, pos_label=0), 76.3, "{:.1f}")

                        predicted negative   precision (neg)   recall (neg)
baseline                             1,075             44.8%          76.3%
30% neg -> pos (mean)                  847             50.3%          67.4%
book: about 845 instead of 1,075 predicted negative; precision 44.8% -> about 50%; recall 76.3% -> about 67%

baseline: predicted negative             this run      1,075   book      1,075   same as the book
baseline: precision (neg), %             this run       44.8   book       44.8   same as the book
baseline: recall (neg), %                this run       76.3   book       76.3   same as the book


Asimetri yalnızca kaç satırın değiştiğiyle mi ilgili? Bölüm her iki yönde aynı *sayıda*
etiket çeviriyor. Burada çekimler tam olarak belirli (kaynak sınıfın satırları üzerinde
`np.random.default_rng(seed).choice`; seed 0, 1, 2), dolayısıyla hücre kitabın
hassasiyetinde karşılaştırıyor.

In [21]:
COUNTS = [250, 500, 1000, 2000]
BOOK_EQUAL = {250: (76.35, 74.98), 500: (76.80, 73.64), 1000: (76.95, 71.11), 2000: (68.60, 67.91)}
print(f"### the same NUMBER of flipped training labels in each direction (seeds {NOISE_SEEDS}, mean macro-F1)")
for count in COUNTS:
    neg_to_pos, pos_to_neg = [], []
    for seed in NOISE_SEEDS:
        rng = np.random.default_rng(seed)
        for rows, out in [(negative_rows, neg_to_pos), (positive_rows, pos_to_neg)]:
            labels = y_train.copy()
            chosen = rng.choice(rows, count, replace=False)
            labels[chosen] = 1 - labels[chosen]
            out.append(macro_f1(y_test, predict_with(labels)))
    print(f"{count:>5} labels: {100 * count / len(negative_rows):.1f}% of the negatives, {100 * count / len(positive_rows):.2f}% of the positives")
    vs_book("   neg -> pos", np.mean(neg_to_pos), BOOK_EQUAL[count][0], "{:.2f}")
    vs_book("   pos -> neg", np.mean(pos_to_neg), BOOK_EQUAL[count][1], "{:.2f}")

### the same NUMBER of flipped training labels in each direction (seeds [0, 1, 2], mean macro-F1)


  250 labels: 9.9% of the negatives, 0.67% of the positives
   neg -> pos                            this run      76.35   book      76.35   same as the book
   pos -> neg                            this run      74.98   book      74.98   same as the book


  500 labels: 19.8% of the negatives, 1.33% of the positives
   neg -> pos                            this run      76.80   book      76.80   same as the book
   pos -> neg                            this run      73.64   book      73.64   same as the book


 1000 labels: 39.5% of the negatives, 2.67% of the positives
   neg -> pos                            this run      76.95   book      76.95   same as the book
   pos -> neg                            this run      71.11   book      71.11   same as the book


 2000 labels: 79.1% of the negatives, 5.34% of the positives
   neg -> pos                            this run      68.60   book      68.60   same as the book
   pos -> neg                            this run      67.91   book      67.91   same as the book


**Nasıl okunur.** Aynı sayıda etiket çevrildiğinde de, 1.000 etikete kadar, yön hâlâ fark
ediyor: fark yalnızca satır sayısıyla açıklanmıyor. 2.000 etikette olumsuz sınıfın beşte
dördü yeniden etiketlenmiş oluyor ve iki yön de düşük skor alıyor. Sonuç bu modele, bu
sınıf oranına ve bu metriğe ait.

## 8. Test etiketleriniz ölçümün anlamını değiştirir

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#test-kümeniz-bir-cetvel-değil-bir-tavan)

Şimdi tersi. Özgün etiketlerle bir kez eğitilmiş tek bir sabit model. Tahminleri
değişmiyor; *test* etiketleri değişiyor: bütün test satırlarının bir payı tekdüze rastgele
çevriliyor ve aynı tahminler bozulmuş etiketlere karşı puanlanıyor.

**Seed'ler.** Bölüm beş seed'in ortalamasını bildiriyor ve hangileri olduğunu basmıyor. Bu
hücre 0 ile 4 arasındaki seed'leri kullanıyor ve bölümün değerinden farkı, bölümün kendi
standart sapmasının yanında basıyor.

**Neye bakacaksınız:** %2 satırı; önceki bölümde bag-of-words'ün özgün etiketlerle aldığı
73,51'in yanında.

In [22]:
BOOK_TEST_NOISE = {0.01: (91.73, 74.68, 53.90, 0.23), 0.02: (90.85, 73.13, 51.31, 0.12), 0.05: (88.31, 69.41, 45.36, 0.17),
                   0.10: (84.19, 64.79, 38.66, 0.24), 0.20: (75.48, 57.00, 28.82, 0.36)}


def noisy_test_labels(rate, seed):
    rng = np.random.default_rng(seed)
    labels = y_test.copy()
    flipped = rng.random(len(labels)) < rate
    labels[flipped] = 1 - labels[flipped]
    return labels


print("### one fixed model, corrupted test labels, mean of 5 seeds (0-4)")
print(f"test noise      acc   macro-F1   F1(neg)   sd(macro-F1)     book macro-F1 (sd)      Δ")
print(f"    0%      {base_acc:7.2f} {base_macro:10.2f} {base_f1n:9.2f}           0.00")
for rate, book in BOOK_TEST_NOISE.items():
    rows = np.array([scores(noisy_test_labels(rate, seed), pred) for seed in range(5)])
    mean = rows.mean(axis=0)
    print(f"{rate:>6.0%}      {mean[0]:7.2f} {mean[1]:10.2f} {mean[2]:9.2f} {rows[:, 1].std(ddof=1):14.2f}"
          f"     {book[1]:>10.2f} ({book[3]:.2f})   {mean[1] - book[1]:+.2f}")

### one fixed model, corrupted test labels, mean of 5 seeds (0-4)
test noise      acc   macro-F1   F1(neg)   sd(macro-F1)     book macro-F1 (sd)      Δ
    0%        92.57      76.21     56.47           0.00
    1%        91.67      74.52     53.61           0.15          74.68 (0.23)   -0.16
    2%        90.80      73.11     51.29           0.08          73.13 (0.12)   -0.02
    5%        88.23      69.46     45.51           0.38          69.41 (0.17)   +0.05
   10%        83.84      64.37     38.04           0.34          64.79 (0.24)   -0.42
   20%        75.33      57.01     28.94           0.22          57.00 (0.36)   +0.01


**Nasıl okunur.** %2'lik test etiketi gürültüsünde aynı model, aynı tahminlerle yaklaşık
üç puan düşük skor alıyor; başka bir modelin özgün etiketlerle aldığı skorun altında.
Bölüm bu karşılaştırmanın ne olduğu konusunda kesin: *farklı etiketlenmiş* iki test
kümesi. Ortak bir test kümesinde sıranın tersine dönmesi değil; farklı etiket
sürümlerinde bildirilen skorlar da doğrudan karşılaştırılamaz.

İki modeli *aynı* gürültülü etiketlere karşı puanlamak başka bir soru ve bölüm onu da
ölçtü: kelime bigram TF-IDF'i ile bag-of-words, her oranda 200 gürültü çekimi. Çekimler
tam olarak belirli (seed 0 ile 199 arası), dolayısıyla hücre kitabın hassasiyetinde
karşılaştırıyor.

In [23]:
bow = CountVectorizer(min_df=2, token_pattern=PAT, **LOC)
pred_bow = clf().fit(bow.fit_transform(train["text"]), y_train).predict(bow.transform(test["text"]))
clean_gap = macro_f1(y_test, pred) - macro_f1(y_test, pred_bow)
vs_book("gap on the original labels", clean_gap, 2.69, "{:.2f}")

N_DRAWS = 200
for rate, book_gap in [(0.02, 2.50), (0.20, 1.36)]:
    gaps = []
    for seed in range(N_DRAWS):
        labels = noisy_test_labels(rate, seed)
        gaps.append(macro_f1(labels, pred) - macro_f1(labels, pred_bow))
    vs_book(f"mean gap at {rate:.0%} shared noise", np.mean(gaps), book_gap, "{:.2f}")
    print(f"   order reversed in {sum(g < 0 for g in gaps)} of {N_DRAWS} draws   (book: none of 200)")

gap on the original labels               this run       2.69   book       2.69   same as the book


mean gap at 2% shared noise              this run       2.50   book       2.50   same as the book
   order reversed in 0 of 200 draws   (book: none of 200)


mean gap at 20% shared noise             this run       1.36   book       1.36   same as the book
   order reversed in 0 of 200 draws   (book: none of 200)


**Nasıl okunur.** Tekdüze ve bağımsız gürültüde fark küçülüyor ve sıra hiç tersine
dönmüyor. Bu, rastgele dağılmış çevirmeler için geçerli. Gerçek etiket hataları iki
modelin ayrıştığı zor satırlarda kümelenebilir; bu yüzden bölüm en kötü durumu da ölçüyor:
iki modelin ayrıştığı satırlara olabilecek en kötü biçimde yerleştirilmiş kaç etiket
değişikliği bir farkı silmeye yetiyor. Arama açgözlü: farkı en çok azaltan tek etiketi
değiştir ve fark kapanana kadar tekrarla.

In [24]:
def changes_to_erase(pred_a, pred_b, limit=400):
    """Açgözlü en kötü durum: a'nın b'ye karşı macro-F1 üstünlüğü bitene kadar ayrışılan satırlarda etiket değişiklikleri."""
    labels = y_test.copy()
    disagree = np.where(pred_a != pred_b)[0]
    changes = 0
    while macro_f1(labels, pred_a) - macro_f1(labels, pred_b) > 0 and changes < limit:
        candidates = [i for i in disagree if pred_a[i] == labels[i]]       # orada a haklı, b yanılıyor
        candidates = candidates[:60] + candidates[-60:]
        best_gap, best_row = None, None
        for i in candidates:
            labels[i] = 1 - labels[i]
            gap = macro_f1(labels, pred_a) - macro_f1(labels, pred_b)
            labels[i] = 1 - labels[i]
            if best_gap is None or gap < best_gap:
                best_gap, best_row = gap, i
        labels[best_row] = 1 - labels[best_row]
        changes += 1
    return changes, len(disagree)


# 4. bölümün 1,2 puanlık çifti: C = 10 ile unigram tf-idf ve bag-of-words
uni = TfidfVectorizer(min_df=2, sublinear_tf=True, token_pattern=PAT, **LOC)
pred_uni = clf(10.0).fit(uni.fit_transform(train["text"]), y_train).predict(uni.transform(test["text"]))

changes, rows = changes_to_erase(pred_uni, pred_bow)
vs_book("1.2-point gap: rows the two disagree on", rows, 231)
vs_book("1.2-point gap: label changes to erase it", changes, 18)
changes, rows = changes_to_erase(pred, pred_bow)
vs_book("2.69-point gap: label changes to erase it", changes, 41)

effects = []
for i in list(np.where(y_test == 0)[0][:150]) + list(np.where(y_test == 1)[0][:150]):
    labels = y_test.copy()
    labels[i] = 1 - labels[i]
    effects.append(abs(macro_f1(labels, pred) - base_macro))
print(f"\none changed label moves one model's macro-F1 by {min(effects):.3f} to {max(effects):.3f} points   (book: 0.02 to 0.045)")

1.2-point gap: rows the two disagree on  this run        231   book        231   same as the book
1.2-point gap: label changes to erase it this run         18   book         18   same as the book


2.69-point gap: label changes to erase it this run         41   book         41   same as the book

one changed label moves one model's macro-F1 by 0.019 to 0.045 points   (book: 0.02 to 0.045)


**Nasıl okunur.** 10.000 etiketin on sekizi, en kötü yerlerde, 1,2 puanlık bir farkı
siliyor. Rastgele dağılmış hatalarda ise fark, önceki hücrenin gösterdiği gibi, ayakta
kalıyor. Demek ki iki model arasındaki küçük bir fark, ayrıştıkları satırlar mümkünse kör
olarak okunmadan ve fark etiketlerin ortak tek bir sürümünde sınanmadan (önceki bölümün
eşleştirilmiş bootstrap'i) bir sonuç değildir.

## 9. Sızıntı: sizi iyi gösteren hatalar

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#sec-sizinti)

Etiket gürültüsü çoğunlukla skoru düşürür. Sızıntı yükseltir. En yaygın biçimi, aynı metnin
bölmenin iki yanında da bulunması; bu yüzden bölüm önce ham dosyadaki yinelemeleri sayıyor,
sonra olağan hatayı bilerek yapıyor: önce böl, hiç tekilleştirme.

**Neye bakacaksınız:** 1.657 fazla satır, 1.657 yineleme grubu demek değil; grupların 77'si
de iki etiketi birden taşıyor.

In [25]:
group_sizes = raw["text"].value_counts()
duplicate_groups = group_sizes[group_sizes > 1]
labels_per_text = raw.groupby("text")["label"].nunique()
vs_book("rows in the raw file", len(raw), 235_165, "{:,}")
vs_book("extra rows removed by de-duplication", len(raw) - raw["text"].nunique(), 1_657, "{:,}")
vs_book("  share of the file, %", 100 * (len(raw) - raw["text"].nunique()) / len(raw), 0.70, "{:.2f}")
vs_book("texts that occur more than once", len(duplicate_groups), 1_055, "{:,}")
vs_book("rows in those groups", int(duplicate_groups.sum()), 2_712, "{:,}")
vs_book("groups that carry both labels", int((labels_per_text > 1).sum()), 77)

# özensiz sürüm: ham dosyayı önce tekilleştirmeden örnekle ve böl
leaky_sub, _ = train_test_split(raw, train_size=50_000, stratify=raw["label"], random_state=42)
leaky_train, leaky_test = train_test_split(leaky_sub, test_size=0.2, stratify=leaky_sub["label"], random_state=42)
leaked = leaky_test["text"].isin(set(leaky_train["text"])).to_numpy()
leaky_vec = make_vec()
leaky_pred = clf().fit(leaky_vec.fit_transform(leaky_train["text"]), leaky_train["label"]).predict(leaky_vec.transform(leaky_test["text"]))
y_leaky = leaky_test["label"].to_numpy()

print("\n### split-before-dedup, the ordinary careless mistake")
vs_book("test rows also present in train", int(leaked.sum()), 45)
vs_book("accuracy on the leaked rows", 100 * accuracy_score(y_leaky[leaked], leaky_pred[leaked]), 93.33, "{:.2f}")
vs_book("accuracy on the other rows", 100 * accuracy_score(y_leaky[~leaked], leaky_pred[~leaked]), 91.90, "{:.2f}")
vs_book("macro-F1 on the leaked rows", macro_f1(y_leaky[leaked], leaky_pred[leaked]), 76.76, "{:.2f}")
vs_book("macro-F1 on the other rows", macro_f1(y_leaky[~leaked], leaky_pred[~leaked]), 74.08, "{:.2f}")
train_label = leaky_train.drop_duplicates("text").set_index("text")["label"]
conflicts = sum(int(train_label[t]) != int(l) for t, l in zip(leaky_test["text"][leaked], y_leaky[leaked]))
vs_book("leaked rows that are positive", int(y_leaky[leaked].sum()), 42)
vs_book("median tokens of a leaked row", int(np.median([len(tokenize(t)) for t in leaky_test["text"][leaked]])), 2)
vs_book("leaked rows whose train copy has the other label", conflicts, 5)

rows in the raw file                     this run    235,165   book    235,165   same as the book
extra rows removed by de-duplication     this run      1,657   book      1,657   same as the book
  share of the file, %                   this run       0.70   book       0.70   same as the book
texts that occur more than once          this run      1,055   book      1,055   same as the book
rows in those groups                     this run      2,712   book      2,712   same as the book
groups that carry both labels            this run         77   book         77   same as the book



### split-before-dedup, the ordinary careless mistake
test rows also present in train          this run         45   book         45   same as the book
accuracy on the leaked rows              this run      93.33   book      93.33   same as the book
accuracy on the other rows               this run      91.90   book      91.90   same as the book
macro-F1 on the leaked rows              this run      76.76   book      76.76   same as the book
macro-F1 on the other rows               this run      74.08   book      74.08   same as the book
leaked rows that are positive            this run         42   book         42   same as the book
median tokens of a leaked row            this run          2   book          2   same as the book
leaked rows whose train copy has the other label this run          5   book          5   same as the book


**Nasıl okunur.** 10.000 satırda sızan kırk beş satır manşet sayıyı oynatamaz; iki altküme
arasındaki 2,68 puanlık fark da ezberin kazancı değil: sızan satırların neredeyse hepsi
olumlu ve iki token uzunluğunda. Beşinde eğitimdeki kopya *diğer* etiketi taşıyor; yani
bir yineleme aynı zamanda etiket sorunlarına bakılacak bir yer. Buradaki 74,08 de 76,21'lik
baseline ile karşılaştırılamaz, çünkü bu farklı bir bölme.

Sırada yakın yinelemeler var: doğru biçimde tekilleştirilmiş bölme; karşılaştırmadan önce
büyük-küçük harf, noktalama ve boşluklar normalleştiriliyor. Bölümün tablosu 34 test
satırı diyor; metinde not edilen kendi yeniden sayımı ise "normalleştirmenin ayrıntısına
göre" 32 veriyor. Bu hücre yeniden sayımın normalleştirmesini kullanıyor.

In [26]:
def normalise(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", tr_lower(s))).strip()


train_normalised = {normalise(s) for s in train["text"]}
near = np.array([(normalise(s) in train_normalised) and normalise(s) != "" for s in test["text"]])
vs_book("near-duplicate test rows", int(near.sum()), 34, note="the chapter's recount gives 32")
vs_book("  of which positive (recount)", int(y_test[near].sum()), 31)
vs_book("  median tokens (recount)", int(np.median(n_tokens[near])), 3)
print("  examples:", ", ".join(repr(s) for s in test["text"][near].head(6)))

near-duplicate test rows                 this run         32   book         34   not comparable: the chapter's recount gives 32
  of which positive (recount)            this run         31   book         31   same as the book
  median tokens (recount)                this run          3   book          3   same as the book
  examples: 'hizli kargo', 'ürün  güzel tavsiye ederim', 'mükemmel***********', 'iyi iş', 'tavsi̇ye ederi̇m,', 'teşekkürler hb.'


Etkinin dozla nasıl büyüdüğünü görmek için bölüm, test satırlarının kontrollü bir payını
eğitim verisine kopyalıyor ve vektörleştiriciyi genişletilmiş kümede yeniden eğitiyor.
%0 ve %100 satırlarında rastgele çekim yok. Diğer dozlar için bölüm çekimini basmıyor;
%10'luk dozun üç çekiminin 76,81 ile 77,13 arasında kaldığını bildiriyor.

**Neye bakacaksınız:** %10 satırı. Gerçek bir modelleme iyileştirmesi büyüklüğünde.

In [27]:
BOOK_DOSE = {0.0: (92.57, 76.21, 56.47), 0.01: (92.57, 76.27, 56.60), 0.02: (92.62, 76.41, 56.86), 0.05: (92.65, 76.53, 57.08),
             0.10: (92.82, 77.14, 58.20), 0.25: (93.33, 78.59, 60.82), 0.50: (94.00, 80.89, 65.07), 1.0: (95.62, 85.89, 74.17)}


def contaminated(share, seed):
    rng = np.random.default_rng(seed)
    copied = rng.choice(len(y_test), int(round(share * len(y_test))), replace=False)
    texts = pd.concat([train["text"], test["text"].iloc[copied]])
    labels = np.concatenate([y_train, y_test[copied]])
    v = make_vec()
    return scores(y_test, clf().fit(v.fit_transform(texts), labels).predict(v.transform(test["text"])))


print("test rows copied into train    acc   macro-F1   F1(neg)     book macro-F1")
for share in BOOK_DOSE:
    if share in (0.0, 1.0):
        result = contaminated(share, 42)
        verdict = "same as the book" if f"{result[1]:.2f}" == f"{BOOK_DOSE[share][1]:.2f}" else "DIFFERS from the book"
        print(f"{share:>26.0%} {result[0]:8.2f} {result[1]:10.2f} {result[2]:9.2f} {BOOK_DOSE[share][1]:>13.2f}   {verdict}")
    else:
        results = np.array([contaminated(share, seed) for seed in (0, 1, 2)])
        mean = results.mean(axis=0)
        print(f"{share:>26.0%} {mean[0]:8.2f} {mean[1]:10.2f} {mean[2]:9.2f} {BOOK_DOSE[share][1]:>13.2f}   "
              f"{mean[1] - BOOK_DOSE[share][1]:+.2f}, three draws {results[:, 1].min():.2f}–{results[:, 1].max():.2f}")

test rows copied into train    acc   macro-F1   F1(neg)     book macro-F1


                        0%    92.57      76.21     56.47         76.21   same as the book


                        1%    92.58      76.29     56.65         76.27   +0.02, three draws 76.28–76.31


                        2%    92.57      76.26     56.59         76.41   -0.15, three draws 76.14–76.34


                        5%    92.71      76.65     57.28         76.53   +0.12, three draws 76.52–76.81


                       10%    92.86      77.13     58.17         77.14   -0.01, three draws 76.95–77.31


                       25%    93.35      78.78     61.20         78.59   +0.19, three draws 78.65–79.01


                       50%    94.10      81.01     65.24         80.89   +0.12, three draws 80.83–81.12


                      100%    95.62      85.89     74.17         85.89   same as the book


**Nasıl okunur.** Bütün test satırlarını eğitime kopyalamak macro-F1'i 76,21'den 85,89'a
çıkarıyor; %10'luk bir bulaşma ise yaklaşık bir puan veriyor: bir modelleme iyileştirmesine
benzeyen ve yalnızca skora bakılarak ondan ayırt edilemeyen bir fark. Etki bu modele, bu
veri kümesine ve bu kopyalama biçimine ait. Her %10 için sabit bir kazanç değil. Bölüm
ayrı ayrı kontrol edilecek altı sızıntı türü sıralıyor; yinelemeler yalnızca ilki.

## 10. Sonraki bin etiketi nereye harcamalı

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#sonraki-bin-etiketi-nereye-harcamalı)

Sabit bir bütçe ya çok sayıda ucuz etiket alır ya da daha az sayıda özenli etiket. Bölümün
ızgarası etiketli yorum sayısını simetrik bir gürültü oranıyla çaprazlıyor.

**Protokol, önceki bölümün öğrenme eğrisinden farklı.** Burada satırlar rastgele çekiliyor
ve vektörleştirici *yalnızca o satırlarda* yeniden eğitiliyor: elinde 1.000 etiketli
yorum olan ve başka metni olmayan biri. Önceki bölümde sözlük ve IDF 40.000 metnin
tamamından geliyordu ve yalnızca sınıflandırıcı daha az etiket görüyordu. İkisi küçük
boyutlarda çok farklı sayılar veriyor; bir tablodaki sayıyı diğerine taşımayın.

**Seed'ler.** Bölüm üç seed'in ortalamasını bildiriyor ve hangileri olduğunu basmıyor. Bu
hücre 0, 1 ve 2 seed'lerini kullanıyor ve farkları basıyor. Küçük hücreler seed'den seed'e
çok oynuyor: bölüm on seed üzerinde, 1.000 etiketli temiz hücre için 2,24 puanlık bir
standart sapma ölçtü.

Hücre 126 eğitim ve yaklaşık yarım dakika sürüyor.

In [28]:
BOOK_GRID = {500: [59.44, 58.56, 55.57, 51.18, 47.40, 42.12], 1_000: [63.71, 60.91, 57.77, 52.32, 46.99, 42.78],
             2_500: [68.76, 64.17, 60.03, 53.36, 48.23, 42.93], 5_000: [71.69, 66.00, 60.21, 53.17, 47.86, 42.42],
             10_000: [73.72, 66.47, 61.81, 54.46, 48.53, 42.88], 20_000: [74.89, 66.99, 62.04, 55.21, 49.09, 43.81],
             40_000: [76.21, 68.18, 63.20, 55.74, 49.34, 44.16]}
GRID_RATES = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40]
GRID_SEEDS = [0, 1, 2]


def budget_cell(n_labels, rate, seed):
    """Rastgele n satır, o satırlarda yeniden eğitilen vektörleştirici, simetrik etiket gürültüsü, özgün test etiketleri."""
    rng = np.random.default_rng(seed)
    if n_labels == len(y_train):
        X, X_eval, labels = Xtr, Xte, y_train.copy()
    else:
        rows = rng.choice(len(y_train), n_labels, replace=False)
        v = make_vec()
        X, X_eval, labels = v.fit_transform(train["text"].iloc[rows]), v.transform(test["text"]), y_train[rows].copy()
    if rate:
        flipped = rng.random(len(labels)) < rate
        labels[flipped] = 1 - labels[flipped]
    return macro_f1(y_test, clf().fit(X, labels).predict(X_eval))


start = time.perf_counter()
grid = {(n_labels, rate): np.mean([budget_cell(n_labels, rate, seed) for seed in GRID_SEEDS])
        for n_labels in BOOK_GRID for rate in GRID_RATES}
print(f"### macro-F1 by budget and label noise (original test labels, seeds {GRID_SEEDS})   ({time.perf_counter() - start:.0f}s)")
print(f"{'':<10}" + "".join(f"{rate:>8.0%}" for rate in GRID_RATES))
for n_labels in BOOK_GRID:
    print(f"{n_labels:>8,}  " + "".join(f"{grid[(n_labels, rate)]:>8.2f}" for rate in GRID_RATES))
    print(f"{'  Δ book':<10}" + "".join(f"{grid[(n_labels, rate)] - b:>+8.2f}" for rate, b in zip(GRID_RATES, BOOK_GRID[n_labels])))

### macro-F1 by budget and label noise (original test labels, seeds [0, 1, 2])   (38s)
                0%      5%     10%     20%     30%     40%
     500     58.25   57.66   56.57   50.96   46.01   42.07
  Δ book     -1.19   -0.90   +1.00   -0.22   -1.39   -0.05
   1,000     63.16   60.69   57.77   52.75   47.98   42.90
  Δ book     -0.55   -0.22   -0.00   +0.43   +0.99   +0.12
   2,500     69.22   63.56   59.26   51.30   46.59   41.60
  Δ book     +0.46   -0.61   -0.77   -2.06   -1.64   -1.33
   5,000     72.30   66.18   61.09   53.55   47.95   43.41
  Δ book     +0.61   +0.18   +0.88   +0.38   +0.09   +0.99
  10,000     73.76   66.77   61.45   54.80   49.00   43.85
  Δ book     +0.04   +0.30   -0.36   +0.34   +0.47   +0.97
  20,000     75.10   68.34   62.91   55.15   49.14   43.85
  Δ book     +0.21   +1.35   +0.87   -0.06   +0.05   +0.04
  40,000     76.21   68.58   62.60   55.41   49.38   43.99
  Δ book     -0.00   +0.40   -0.60   -0.33   +0.04   -0.17


**Nasıl okunur.** Bir sütunu aşağı doğru okuyun: daha çok etiket yardımcı oluyor. Bir
satırı yana doğru okuyun: gürültü, hacmin kazandırdığından fazlasına mal oluyor; 2.500
etiketteki temiz sütun, 40.000 etiketteki %5 gürültülü sütuna yakın. Bölümün hücrelerinden
farklar en çok küçük temiz hücrelerde; üç seed'lik ortalamaların en az kararlı olduğu yer
de orası.

Bölümün metni iki hücre çiftini karşılaştırıyor ve bunları her biri **on** seed ile
yeniden koşturdu, çünkü üç seed'lik yarım puanlık bir fark hiçbir şeyi karara bağlamaz.
Seed'ler (0 ile 9 arası) belirli, dolayısıyla bu karşılaştırma kitabın hassasiyetinde.

In [29]:
BOOK_TEN_SEEDS = {(500, 0.0): 59.20, (1_000, 0.0): 62.24, (2_500, 0.0): 69.26,
                  (40_000, 0.05): 68.38, (40_000, 0.10): 62.66, (40_000, 0.20): 55.32}
ten = {cell: np.array([budget_cell(cell[0], cell[1], seed) for seed in range(10)]) for cell in BOOK_TEN_SEEDS}
for cell, book in BOOK_TEN_SEEDS.items():
    vs_book(f"{cell[0]:>6,} labels, {cell[1]:.0%} noise: mean", ten[cell].mean(), book, "{:.2f}")
    print(f"      sd {ten[cell].std(ddof=1):.2f}, range {ten[cell].min():.2f}–{ten[cell].max():.2f}")
print()
for a, b, book_ahead in [((1_000, 0.0), (40_000, 0.10), 37), ((500, 0.0), (40_000, 0.20), 100)]:
    ahead = int((ten[a][:, None] > ten[b][None, :]).sum())
    vs_book(f"{a[0]:,} clean ahead of 40,000 at {b[1]:.0%}, of 100 pairs", ahead, book_ahead)

   500 labels, 0% noise: mean            this run      59.20   book      59.20   same as the book
      sd 1.19, range 57.57–61.13
 1,000 labels, 0% noise: mean            this run      62.24   book      62.24   same as the book
      sd 2.24, range 59.29–65.44
 2,500 labels, 0% noise: mean            this run      69.26   book      69.26   same as the book
      sd 1.27, range 67.02–71.14
40,000 labels, 5% noise: mean            this run      68.38   book      68.38   same as the book
      sd 0.48, range 67.61–69.28
40,000 labels, 10% noise: mean           this run      62.66   book      62.66   same as the book
      sd 0.46, range 61.92–63.27
40,000 labels, 20% noise: mean           this run      55.32   book      55.32   same as the book
      sd 0.47, range 54.68–56.14

1,000 clean ahead of 40,000 at 10%, of 100 pairs this run         37   book         37   same as the book
500 clean ahead of 40,000 at 20%, of 100 pairs this run        100   book        100   same as the book


**Nasıl okunur.** On seed üzerinde 1.000 temiz etiket ile %10 gürültülü 40.000 etiket ayırt
edilemiyor: temiz hücre seed çiftlerinin yaklaşık üçte birinde önde ve kendi seed'ler
arası yayılımı farktan büyük. 500 temiz etiket ile %20 gürültülü 40.000 etiket ise başka
bir durum; temiz hücre her çiftte önde.

5.000 temiz etiketin 40.000 gürültülü etikete eşit olduğu başa baş oranı ızgarada yok.
Bölüm bu oranı, iki yanındaki ölçülmüş iki hücre arasına çekilen düz bir çizgiden okuyor
ve ara değerleme diye etiketliyor.

In [30]:
clean_5000, at_0, at_5 = BOOK_GRID[5_000][0], BOOK_GRID[40_000][0], BOOK_GRID[40_000][1]
break_even = 0.05 * (at_0 - clean_5000) / (at_0 - at_5)
print(f"5,000 clean labels: {clean_5000}; 40,000 labels: {at_0} at 0% noise, {at_5} at 5%")
vs_book("break-even noise rate, interpolated, %", 100 * break_even, 2.8, "{:.1f}")
print("an interpolation between two measured cells of the chapter's grid, not a measured threshold")

5,000 clean labels: 71.69; 40,000 labels: 76.21 at 0% noise, 68.18 at 5%
break-even noise rate, interpolated, %   this run        2.8   book        2.8   same as the book
an interpolation between two measured cells of the chapter's grid, not a measured threshold


### Her etiket aynı değerde değil

[Bölümdeki kısım](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#her-etiket-aynı-değerde-değil)

Hangi satırların etiketleneceğini seçebiliyorsanız, mevcut modelin en az emin olduklarını
seçin. Bölümün şeması: rastgele 200 etiketle eğit, etiketsiz havuzu puanla ve puanı 0,5'e
en yakın 800 örneği al.

Bölüm tek bir tablo basıyor (rastgele 1.000: 62 olumsuzla 65,49 macro-F1; 200 + 800
belirsiz: 248 olumsuzla 68,80) ve ardından, açıkça tarif ettiği bir protokolle yaptığı
kendi yeniden koşusunun üçer seed'lik iki grupta 5,3 ve 5,2 puanlık kazanç verdiğini
söylüyor. Bu hücre o tarif edilen protokolü uyguluyor: başlangıç modeli 40.000 metnin
vektörleştiricisini okuyor, son modelin vektörleştiricisi ise seçilen 1.000 satırda
eğitiliyor. 0, 1 ve 2 seed'leri bölümün iki grubundan biri.

**Neye bakacaksınız:** her stratejinin kaç olumsuz yorum bulduğu.

In [31]:
def spend_two_ways(seed):
    order = np.random.default_rng(seed).permutation(len(y_train))
    random_rows, seed_rows, pool = order[:1_000], order[:200], order[200:]
    # bölümün şeması: rastgele 200 etiket, sonra havuzdan puanı 0,5'e en yakın 800 örnek
    m0 = clf().fit(Xtr[seed_rows], y_train[seed_rows])
    unc = np.abs(m0.predict_proba(Xtr[pool])[:, 1] - 0.5)
    picks = pool[np.argsort(unc)[:800]]
    out = {}
    for name, rows in [("random", random_rows), ("uncertain", np.concatenate([seed_rows, picks]))]:
        v = make_vec()
        p = clf().fit(v.fit_transform(train["text"].iloc[rows]), y_train[rows]).predict(v.transform(test["text"]))
        out[name] = (int((y_train[rows] == 0).sum()),) + scores(y_test, p)
    return out


AL_SEEDS = [0, 1, 2]
runs = [spend_two_ways(seed) for seed in AL_SEEDS]
print(f"### 1,000 labels, spent two ways (seeds {AL_SEEDS})")
print(f"{'':<30}negatives found    acc   macro-F1   F1(neg)")
means = {}
for name, label in [("random", "1,000 drawn at random"), ("uncertain", "200 random + 800 uncertain")]:
    means[name] = np.mean([run[name] for run in runs], axis=0)
    print(f"{label:<30}{means[name][0]:>15.0f} {means[name][1]:>6.2f} {means[name][2]:>10.2f} {means[name][3]:>9.2f}")
print()
vs_book("random 1,000: macro-F1", means["random"][2], 63.13, "{:.2f}")
vs_book("200 + 800 uncertain: macro-F1", means["uncertain"][2], 68.46, "{:.2f}")
vs_book("gain, points", means["uncertain"][2] - means["random"][2], 5.3, "{:.1f}")
vs_book("negatives found, uncertain", means["uncertain"][0], 290, "{:.0f}")

### 1,000 labels, spent two ways (seeds [0, 1, 2])
                              negatives found    acc   macro-F1   F1(neg)
1,000 drawn at random                      60  93.16      63.13     29.85
200 random + 800 uncertain                290  93.75      68.46     40.22

random 1,000: macro-F1                   this run      63.13   book      63.13   same as the book
200 + 800 uncertain: macro-F1            this run      68.46   book      68.46   same as the book
gain, points                             this run        5.3   book        5.3   same as the book
negatives found, uncertain               this run        290   book        290   same as the book


**Nasıl okunur.** Belirsizlik örneklemesi birkaç kat fazla olumsuz yorum buluyor ve aynı
1.000 etiketle daha yüksek skor alıyor. Yön, bölümün basılı tablosundakiyle aynı; büyüklük
protokole ve seed'e bağlı, bölümün ikisini de vermesinin sebebi bu. Yanında üç sınır
geliyor. Belirsizlikle seçilmiş bir küme doğal dağılımı temsil etmez; bir *test* kümesini
asla böyle kurmayın. Seçim metinlerin zorluğunu da değiştirir; kazanç yalnızca olumsuz
sayısıyla açıklanmaz. Ve küme, onu seçen modele bağlıdır.

## 11. Alıştırmalar

Bunlar [5. bölümün kendi alıştırmalarını](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/tr/chapters/05-veri-etiketleme.html#alıştırmalar) yukarıdaki kodla sürdürüyor. Köşeli parantezdeki sayılar
bölümdeki alıştırma numaraları.

1. **Tek bir ham uyum için iki tablo** [2]. 5. kısımdaki `coefficients` ile, 100 öğenin
   90'ında uyuşan iki etiketleyici için 2×2 tabloyu kurun: önce ikisi de olumlu etiketi 50
   kez, sonra ikisi de 95 kez kullanıyor. Marjinallerin ve 90 uyumun her durumda yalnızca
   tek bir tabloya izin verdiğini gösterin. Sonra ham uyumu %90'da tutup marjinalleri
   değiştirin ve kappa'nın nasıl oynadığını izleyin.
2. **Kendi denetiminiz** [3]. 3. kısımdaki seed'i değiştirip başka bir yüz satır okuyun.
   *Hata*, *karara bağlanamaz* ve *tamam* ölçütlerinizi başlamadan önce yazın. Hiç açık
   hata bulmazsanız üç kuralını, bulursanız `wilson(k, 100)` sonucunu bildirin.
3. **Dengeli bir derlem** [4]. `train` içinden sınıf açısından dengeli bir eğitim kümesi
   kurun ve 7. kısmın iki hücresini yeniden çalıştırın. İki yön arasındaki asimetri ayakta
   kalıyor mu? Sonucu sınıf sayılarından açıklayın.
4. **Kendi ikinci etiketleyiciniz** [5]. 5. kısımdaki `NEG` ve `POS` listelerini kendi
   göreviniz için yazdığınız bir kuralla ya da bir düzenli ifadeyle değiştirin. Kappa'yı
   tam test kümesinde ve dengelenmiş altörneklemde, 2×2 tablo ve iki değerlendiricinin
   oranlarıyla birlikte bildirin.
5. **Out-of-fold, tam anlamıyla** [6]. 6. kısmın kat içi puanlarını (`wrong_conf_infold`)
   kullanın. En yüksek puanlı yirmi adayı saklanan etikete bakmadan okuyun. Sonra üç
   koşulu test bölmesinde karşılaştırın: silmek, çevirmek, yalnızca elle doğruladıklarınızı
   onarmak. Her biri için rastgele satır kontrolünü koşturun.
6. **Başka bir bütçe için başa baş** [8]. 10. kısımdaki `BOOK_GRID` üzerinden, 2.500 temiz
   etiketin 40.000 gürültülü etikete hangi iki ölçülmüş gürültü düzeyi arasında eşit
   olduğunu bulun, ara değerlenmiş oranı hesaplayın ve ara değerleme diye etiketleyin.
   Sonra tablonun, bir bütçeyi eğitim ve test etiketleri arasında bölmeye karar vermek için
   neyi içermediğini söyleyin.
7. **Çelişen yinelemeleri okuyun** [9]. 9. kısım iki etiketi birden taşıyan 77 yineleme
   grubu saydı. Bunları
   `raw[raw["text"].isin(labels_per_text[labels_per_text > 1].index)]` ile listeleyin,
   okuyun ve açık hata, eksik bağlam ve şema belirsizliğini ayırın. Çelişki oranlarının
   rastgele-100 denetimiyle neden karşılaştırılamayacağını bir cümleyle söyleyin.
8. **Başka bir model için belirsizlik örneklemesi** [bölümün üçüncü sınırı].
   `spend_two_ways` içinde TF-IDF başlangıç modelinin seçtiği satırları koruyun, ama son
   modeli başka bir temsille eğitin; örneğin önceki bölümdeki karakter n-gram'larıyla.
   Seçilen küme rastgele kümeyi hâlâ geçiyor mu?

Bölümün sonundaki uygulama kod gerektirmiyor: iki kişi, 100 yorumluk iki dengeli tur,
turlar arasında yazılan bir kılavuz ve kendi etiketleriniz üzerinde 5. kısmın uyum
ölçüleri.

---

*Bu notebook'taki kod MIT lisanslı; anlatım CC BY 4.0. Türkçe ürün yorumları veri kümesi
CC BY-SA 4.0 (Fatih Barmanbay, Hugging Face'te `fthbrmnby/turkish_product_reviews`); bu
notebook'un çıktılarında basılan yorum metinleri o veri kümesinden geliyor.*